# Fine-tuning an LLMON model with LoRA — tau-bench next-action


## Step 0 — Environment

In [1]:
import json, os, random, sys, time
import torch, transformers, peft, trl, datasets
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
print("transformers", transformers.__version__, "| peft", peft.__version__,
      "| trl", trl.__version__, "| datasets", datasets.__version__)
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

# Requires: pip install -r requirements.txt (this directory), and pip install -e ../../.. (the repository, for llmon.lang)
from llmon.lang.json.llmon_converters import LLMONConverters

torch 2.7.0+cu126 cuda True
transformers 5.9.0 | peft 0.20.0 | trl 1.9.2 | datasets 5.0.1
gpu: NVIDIA H200


## Step 1 — Point at the model

In [2]:
MODEL_PATH = os.environ.get("LLMON_MODEL", "ibm-granite/granite-4.1-3b-llmon-preview")
REVISION = "273198fbfd138f2a33506bb47fd7dfc4b2189d7a"  # bf16 weights (7.3 GB); the latest revision stores float32 (14.6 GB)
OUT_ROOT   = os.environ.get("LLMON_OUT", "./finetune_out")
TAG        = "taubench-nextaction"

tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_PATH, revision=REVISION)

DELIMS = ["<|open|>", "<|close|>", "<|open_end|>", "<|:|>", "<|.|>", "<|self_close|>"]
ids = {d: tokenizer.convert_tokens_to_ids(d) for d in DELIMS}
assert all(v is not None and v != tokenizer.unk_token_id for v in ids.values()), \
    f"model cannot read LLMON: {ids}"
print("model:", MODEL_PATH)
print("output:", OUT_ROOT)
print("delimiters are single tokens:", ids)

model: ibm-granite/granite-4.1-3b-llmon-preview
output: ./finetune_out
delimiters are single tokens: {'<|open|>': 100284, '<|close|>': 100286, '<|open_end|>': 100285, '<|:|>': 100289, '<|.|>': 100288, '<|self_close|>': 100287}


## Step 2 — Load the dataset from the Hub

In [3]:
from datasets import load_dataset
ds = load_dataset("amityco/tau-bench-retail-train-next-action", split="train")
print(ds)
row = ds[0]
print(f"\n{len(row['conversations'])} history messages; roles:",
      [m["role"] for m in row["conversations"]])
tc = row["answer"][0]["tool_calls"][0]["function"]
print("target tool call:", tc["name"], tc["arguments"][:120])

Dataset({
    features: ['conversations', 'answer'],
    num_rows: 82
})

13 history messages; roles: ['user', 'assistant', 'user', 'assistant', 'tool', 'assistant', 'tool', 'assistant', 'tool', 'assistant', 'user', 'assistant', 'user']
target tool call: return_delivered_order_items {"order_id": "#W6067464", "item_ids": ["8917609800", "9624127908"], "payment_method_id": "credit_card_4190576"}


## Step 3 — Convert to LLMON


**Anything the model must reason over is a labelled value, not a positional
   convention.** Each history message becomes labelled fields (`role`, `content`,
   `tool_name`, `tool_arguments`) rather than one flat string.

`no_openend_tagname=True` gives the bare `<|open_end|>` close this model was trained with;
the older `<|open_end|>TAG<|close|>` form is not model-native.

In [4]:
_conv = LLMONConverters(converting_to_json=False, with_mellea=False, keys_as_item=False,
                        no_expand_tags=False, no_openend_tagname=True,
                        no_composite_tags=True)
def llmonize(obj):
    return _conv.json_dict_to_llmon_str(obj, compact=True)

# OUR framing: this dataset carries no system message, so instr/req are supplied here.
INSTR = ("You are a retail support agent. Given the conversation so far, produce the "
         "single next tool call and nothing else.")
REQS = [
    {"name": "Next_action_only", "says": "Emit exactly one tool call. Do not narrate."},
    {"name": "Grounded_arguments", "says": "Every argument value must come from the "
                                           "conversation, never from prior knowledge."},
]

def msg_obj(m):
    # one history message as labelled fields, never a flat string
    o = {"role": m["role"]}
    if m.get("content"):
        o["content"] = m["content"]
    for c in (m.get("tool_calls") or []):
        o["tool_name"] = c["function"]["name"]
        o["tool_arguments"] = c["function"]["arguments"]   # a JSON string in this dataset
    return o

def target_obj(answer):
    f = answer[0]["tool_calls"][0]["function"]
    args = f["arguments"]
    if isinstance(args, str):
        args = json.loads(args)          # -> dict, so LLMON renders real structure
    return {"tool_call": {"name": f["name"], "arguments": args}}

def render(row):
    msgs = [{
        "role": "user",
        "instruction": [{"value": INSTR, "exec": True}],
        "data": [{"value": llmonize(msg_obj(m)), "exec": True}
                 for m in row["conversations"]],
        "requirement": [{"value": llmonize(r), "exec": True} for r in REQS],
    }]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return prompt, llmonize(target_obj(row["answer"])) + "<|open_end|>"

pairs = []
for row in ds:
    try:
        p, c = render(row)
    except (KeyError, IndexError, json.JSONDecodeError) as e:
        print("  skipped a row:", type(e).__name__, e)
        continue
    pairs.append({"prompt": p, "completion": c + tokenizer.eos_token})
print(f"rendered {len(pairs)} of {len(ds)} rows")

# The answer must not appear in the prompt. Check TAG-shaped, not word-shaped: tool names
# and ids legitimately recur in the history, so a substring search false-positives.
leaks = [i for i, p in enumerate(pairs) if "<|open|>tool_call" in p["prompt"]]
assert not leaks, f"target block leaked into {len(leaks)} prompts"
print("leak guard passed")

print("\n--- prompt (last 700 chars) ---\n" + pairs[0]["prompt"][-700:])
print("\n--- completion ---\n" + repr(pairs[0]["completion"]))
MAX_LENGTH = 12288
lens = [len(tokenizer(p["prompt"] + p["completion"])["input_ids"]) for p in pairs]
print(f"\ntokens per example: median {sorted(lens)[len(lens)//2]}  max {max(lens)}")
# Right-sided truncation eats the completion, so an example longer than the model
# context would train on a prompt with no answer. Fail loudly rather than silently.
assert max(lens) <= MAX_LENGTH, (
    f"{sum(1 for x in lens if x > MAX_LENGTH)} examples exceed {MAX_LENGTH} tokens "
    f"(max {max(lens)}); raise max_length rather than letting them truncate")
print(f"all {len(pairs)} examples fit in {MAX_LENGTH} tokens -- none dropped, none truncated")

rendered 82 of 82 rows
leak guard passed

--- prompt (last 700 chars) ---
xec<|.|>input<|close|>data<|:|>üõ<|open_end|><|open|>exec<|.|>input<|close|>data<|:|>Úñ<|open_end|><|open|>exec<|.|>input<|close|>data<|:|>«Ñ<|open_end|><|open|>exec<|.|>input<|close|>data<|:|>¡¾<|open_end|><|open|>exec<|.|>input<|close|>data<|:|>¥ô<|open_end|><|open|>exec<|.|>input<|close|>data<|:|>««<|open_end|><|open|>exec<|.|>input<|close|>data<|:|>üÜ<|open_end|><|open|>exec<|.|>input<|close|>data<|:|>©Ú<|open_end|><|open|>exec<|.|>input<|close|>data<|:|>Îé<|open_end|><|open|>exec<|.|>requirement<|close|>req<|:|>×«<|open_end|><|open|>exec<|.|>requirement<|close|>req<|:|>º§<|open_end|><|open|>exec<|.|>output<|close|>assistant<|:|>ðê<|open_end|><|open_end|><|open|>assistant<|:|>ðê<|close|>

--- completion ---
'<|open|>tool_call<|.|>name<|close|>return_delivered_order_items<|open_end|><|open|>tool_call<|.|>arguments<|.|>order_id<|close|>#W6067464<|open_end|><|open|>tool_call<|.|>arguments<|.|>item_ids<|:|>0<|clo


tokens per example: median 2820  max 9244
all 82 examples fit in 12288 tokens -- none dropped, none truncated


## Step 4 — Split


In [5]:
random.Random(42).shuffle(pairs)
n_val = max(4, int(len(pairs) * 0.1))
val, train = pairs[:n_val], pairs[n_val:]
print(f"train={len(train)}  val={len(val)}")

train=74  val=8


## Step 5 — LoRA and training arguments


In [6]:
from datasets import Dataset
model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, revision=REVISION, dtype=torch.bfloat16, device_map="auto")     # dtype=, not torch_dtype=

lora = peft.LoraConfig(task_type="CAUSAL_LM", r=32, lora_alpha=64, lora_dropout=0.05,
                       bias="none", target_modules="all-linear")
args = trl.SFTConfig(
    output_dir=f"{OUT_ROOT}/{TAG}-out",
    num_train_epochs=6, learning_rate=2e-4,   # 74 examples = ~10 steps/epoch; at 3 epochs
                                              # val token accuracy is only 0.93, at 6 it is 1.0
    per_device_train_batch_size=2, gradient_accumulation_steps=4,
    max_length=12288,                # raised from 8192: the longest conversation is
                                 # 9,166 tokens, and truncation is right-sided so
                                 # it would eat the completion. Nothing dropped.
    completion_only_loss=True,
    loss_type="nll",
    router_aux_loss_coef=0.0,
    bf16=True, gradient_checkpointing=True,
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=3,
    load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
    logging_steps=5, report_to=[],
)
trainer = trl.SFTTrainer(
    model=model, args=args, peft_config=lora,
    train_dataset=Dataset.from_list(train), eval_dataset=Dataset.from_list(val),
)
trainer.model.print_trainable_parameters()

Adding EOS to train dataset:   0%|          | 0/74 [00:00<?, ? examples/s]

Adding EOS to train dataset: 100%|██████████| 74/74 [00:00<00:00, 14322.96 examples/s]

Tokenizing train dataset:   0%|          | 0/74 [00:00<?, ? examples/s]

Tokenizing train dataset:  16%|█▌        | 12/74 [00:00<00:00, 107.03 examples/s]

Tokenizing train dataset:  36%|███▋      | 27/74 [00:00<00:00, 95.23 examples/s] 

Tokenizing train dataset:  50%|█████     | 37/74 [00:00<00:00, 93.02 examples/s]

... (19 outputs left out)

Truncating eval dataset: 100%|██████████| 8/8 [00:00<00:00, 473.34 examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset: 100%|██████████| 8/8 [00:00<00:00, 561.51 examples/s]

trainable params: 62,259,200 || all params: 3,465,095,680 || trainable%: 1.7968


## Step 6 — Train

In [7]:
trainer.train()

Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,1.207290,1.055206,0.888574,0.723618,237687.000000
2,0.685827,0.583526,0.821419,0.731993,475374.000000
3,0.500869,0.477288,0.848006,0.922948,713061.000000
4,0.436274,0.423254,0.811233,0.932998,950748.000000
5,0.403526,0.394695,0.800818,0.994975,1188435.000000
6,0.387567,0.386835,0.811752,0.996650,1426122.000000


TrainOutput(global_step=60, training_loss=0.7522228022416433, metrics={'train_runtime': 376.1312, 'train_samples_per_second': 1.18, 'train_steps_per_second': 0.16, 'total_flos': 3.539533044323328e+16, 'train_loss': 0.7522228022416433, 'epoch': 6.0})

## Step 7 — Save


In [8]:
trainer.model.save_pretrained(f"{OUT_ROOT}/{TAG}-adapter")
merged = trainer.model.merge_and_unload()
merged.save_pretrained(f"{OUT_ROOT}/{TAG}-hf")
tokenizer.save_pretrained(f"{OUT_ROOT}/{TAG}-hf")
print("saved adapter + merged model")

merged.eval()
merged.config.use_cache = True
try:
    merged.gradient_checkpointing_disable()
except Exception:
    pass

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:07<00:00,  7.66s/it]

Writing model shards: 100%|██████████| 1/1 [00:07<00:00,  7.66s/it]

saved adapter + merged model


## Step 8 — Sanity check on a held-out example

In [9]:
ex = val[0]
enc = tokenizer(ex["prompt"], return_tensors="pt", add_special_tokens=False).to(merged.device)
with torch.no_grad():
    # The checkpoint's generation_config lists <|open_end|> as a stop token, but every field of
    # the answer closes with a bare <|open_end|>, so the default would stop after the first
    # field. Stop on <|end_of_text|> only.
    out = merged.generate(**enc, max_new_tokens=256, do_sample=False,
                          eos_token_id=tokenizer.eos_token_id,
                          pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
gen = tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=False)
want = ex["completion"].replace(tokenizer.eos_token, "").strip()
print("expected :", repr(ex["completion"]))
print("generated:", repr(gen))
print("\nexact match:", gen.strip().startswith(want))

expected : '<|open|>tool_call<|.|>name<|close|>return_delivered_order_items<|open_end|><|open|>tool_call<|.|>arguments<|.|>order_id<|close|>#W5564375<|open_end|><|open|>tool_call<|.|>arguments<|.|>item_ids<|:|>0<|close|>7583936705<|open_end|><|open|>tool_call<|.|>arguments<|.|>item_ids<|:|>1<|close|>1775591963<|open_end|><|open|>tool_call<|.|>arguments<|.|>payment_method_id<|close|>paypal_2299608<|open_end|><|open_end|><|end_of_text|>'
generated: '<|open|>tool_call<|.|>name<|close|>return_delivered_order_items<|open_end|><|open|>tool_call<|.|>arguments<|.|>order_id<|close|>#W5564375<|open_end|><|open|>tool_call<|.|>arguments<|.|>item_ids<|:|>0<|close|>7583936705<|open_end|><|open|>tool_call<|.|>arguments<|.|>item_ids<|:|>1<|close|>1775591963<|open_end|><|open|>tool_call<|.|>arguments<|.|>payment_method_id<|close|>paypal_2299608<|open_end|><|open_end|><|end_of_text|>'

exact match: True
